In [21]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv, find_dotenv
from typing import TypedDict, Literal
from langchain_openai import ChatOpenAI
from langchain.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field
import os

In [22]:
load_dotenv(find_dotenv())

True

In [23]:
generator_llm = ChatOpenAI(
    model="gpt-4o-mini",
    base_url="https://models.inference.ai.azure.com",
    api_key=os.getenv("GITHUB_TOKEN")
)

optimiser_llm = ChatOpenAI(
    model="gpt-4o",
    base_url="https://models.inference.ai.azure.com",
    api_key=os.getenv("GITHUB_TOKEN")
)

evaluator_llm = ChatOpenAI(
    model="gpt-4o-mini",
    base_url="https://models.inference.ai.azure.com",
    api_key=os.getenv("GITHUB_TOKEN")
)

In [24]:
class TweetEvaluation(BaseModel):
    evaluation: Literal["Approved", "Needs Improvement"] = Field(..., description="The evaluation result of the tweet.")
    feedback: str = Field(..., description="Feedback provided for the tweet.")

In [25]:
class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation: Literal["Approved", "Needs Improvement"]
    feedback: str
    iteration: int
    max_iterations: int

In [26]:
def generate_tweet(state: TweetState):
    messages = [
        SystemMessage(content="You are a funny and clever twitter/X influencer."),
        HumanMessage(content=f"""
Write a short, original, and witty tweet about the topic: {state['topic']}. The tweet should be less than 280 characters and should be engaging and humorous.

Rules:
Do NOT use question-answer format, and do NOT include hashtags or emojis.
Use simple, clear language that is easy to understand. Avoid using complex words or jargon. The tweet should be suitable for a general audience and should not contain any offensive or inappropriate content.
This is the version {state['iteration']} of the tweet, and you have a maximum of {state['max_iterations']} iterations to improve it.
"""),
    ]
    response = generator_llm.invoke(messages).content
    return {'tweet': response}

In [27]:
def evaluate_tweet(state: TweetState):
    messages = [
        SystemMessage(content="You are a ruthless and no-nonsense twitter critic. You evaluate tweets based on their humor, originality, and engagement potential."),
        HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality - Is this fresh, or have you seen it a hundred times before?
2. Humor - Did it genuinely make you smile, laugh, or chuckle?
3. Punchiness - Is it short, sharp, and scroll-stopping?
4. Virality Potential - Would people retweet or share it?
5. Format - Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Don't end with generic, throwaway, or deflating lines that weaken the humor (e.g., "Masterpieces...")

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"
- feedback: One paragraph explaining the strengths and weaknesses
""")]

    structured_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluation)
    response = structured_evaluator_llm.invoke(messages)
    return {'evaluation': response.evaluation, 'feedback': response.feedback}

In [28]:
def improve_tweet(state: TweetState):
    messages = [
        SystemMessage(content="You are a witty and clever twitter influencer. You improve tweets based on feedback."),
        HumanMessage(content=f"""
The current tweet is: "{state['tweet']}"
Feedback: "{state['feedback']}"
Please improve the tweet based on the feedback provided. Make it more original, humorous, and engaging. Ensure it is less than 280 characters, does not use question-answer format, and does not include hashtags or emojis. Keep the language simple and clear, suitable for a general audience. This is the version {state['iteration']} of the tweet, and you have a maximum of {state['max_iterations']} iterations to improve it.
"""),
    ]
    response = optimiser_llm.invoke(messages).content
    iteration = state['iteration'] + 1
    return {'tweet': response, 'iteration': iteration}

In [29]:
def route_evaluation(state: TweetState):
    if state['evaluation'] == "Approved" or state['iteration'] >= state['max_iterations']:
        return "approved"
    else:
        return "needs_improvement"

In [30]:
graph = StateGraph(TweetState)

graph.add_node("generate", generate_tweet)
graph.add_node("evaluate", evaluate_tweet)
graph.add_node("improve", improve_tweet)

graph.add_edge(START, "generate")
graph.add_edge("generate", "evaluate")
graph.add_conditional_edges("evaluate", route_evaluation, {"approved": END, "needs_improvement": "improve"})
graph.add_edge("improve", "evaluate")

workflow = graph.compile()

In [33]:
initial_state = {
    "topic": "Indian politics",
    "iteration": 1,
    "max_iterations": 3,
}

workflow.invoke(initial_state)

{'topic': 'Indian politics',
 'tweet': 'Indian politics is like a Bollywood movie: full of drama, unexpected plot twists, and a shocking amount of singing about how great the hero is—while the side characters are wondering when they get their time in the spotlight.',
 'evaluation': 'Approved',
 'feedback': "This tweet showcases a strong mix of originality and humor, drawing a creative parallel between Indian politics and Bollywood movies. The metaphor is fresh and captures attention with its relatable content. The punchiness is effective, making it easily digestible and potentially engaging for a broad audience. It also highlights the often-overlooked side characters, adding depth to the humor. Overall, it's well-structured and under 280 characters, making it a solid candidate for sharing and retweeting.",
 'iteration': 1,
 'max_iterations': 3}